# DeepAR Forecast Demo (synthetic AR data)

A minimal end-to-end demo of the `tfts` DeepAR model: generate the synthetic AR series, build windows with the `tfts` DeepAR pipeline (`x` = lookback window, `decoder_feature` = teacher-forced lagged target, `static` = series id, `y` = forecast), train a probabilistic autoregressive LSTM with a negative-log-likelihood (Gaussian) loss, then produce a point forecast by **ancestral sampling** (mean over 100 sampled predictive paths) and plot it.


Model config (defaults of `AutoConfig.for_model("deep_ar")`): LSTM `hidden_size=30`, `rnn_layers=2`, series `embedding_size=50`, `n_series=100`, dropout `0.1`.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath("../.."))  # make the local ./tfts package importable

import math

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf

tf.random.set_seed(42)
np.random.seed(42)

ENCODER_LENGTH = 60  # lookback (context) window
PREDICTION_LENGTH = 20  # forecast horizon
BATCH_SIZE = 128
LOG2PI = math.log(2.0 * math.pi)

## 1. Data
Generate the synthetic AR data (quadratic trend + seasonality + noise, 100 series x 400 steps) using the same `generate_ar_data(...)` call and seed as the tutorial.

In [ ]:
from tfts.data.ar import generate_ar_data

data = generate_ar_data(seasonality=10.0, timesteps=400, n_series=100, seed=42)
print(data.head())
print(
    "rows:",
    len(data),
    "| series:",
    data.series.nunique(),
    "| time range:",
    data.time_idx.min(),
    "-",
    data.time_idx.max(),
)

## 2. Windowing (DeepAR pipeline)
Use `ARDeepARPreprocessor`, which replicates the Phase 1 reference pipeline:
- the same train/validation split (`training_cutoff = max_time_idx - 20`) and encoder/decoder lengths (60/20),
- `value` as the only time-varying real (normalized), `series` carried as a static categorical (embedding id),
- a **teacher-forced** `decoder_feature` = the shifted/lagged target `[last_encoder_value, y[0], ..., y[-2]]`,
- a target normalizer fit on the training `value` (global, `EncoderNormalizer`-style before a `transformation`).

In [ ]:
from tfts.data.ar import ARDeepARPreprocessor

# global target normalizer fit on the training portion (mirrors PF EncoderNormalizer)
cutoff = data.time_idx.max() - PREDICTION_LENGTH
train_vals = data.loc[data.time_idx <= cutoff, "value"]
mean, std = float(train_vals.mean()), float(train_vals.std())
print("normalizer  mean:", round(mean, 4), " std:", round(std, 4))

proc = ARDeepARPreprocessor(
    data, encoder_length=ENCODER_LENGTH, prediction_length=PREDICTION_LENGTH, mean=mean, std=std
)
train_batch = proc.train()  # sliding windows fully in time_idx <= training_cutoff
val_batch = proc.validation()  # one forecast per series starting at training_cutoff+1

print("train x:", train_batch.x.shape, "  y:", train_batch.y.shape)
print("train decoder_feature:", train_batch.decoder_feature.shape, "  static:", train_batch.static.shape)
print("val   x:", val_batch.x.shape, "  y:", val_batch.y.shape)
print("teacher-forced decoder_feature[0] (first 3):", train_batch.decoder_feature[0, :3, 0])

## 3. Build the model
Use the `tfts` AutoModel registry (the DeepAR implementation lives in `tfts/models/deep_ar.py`). It takes the 3-tuple `(x, decoder_feature, static)` — encoder window, teacher-forced lagged target, and the series-id static input.

In [ ]:
from tfts.models.auto_config import AutoConfig
from tfts.models.auto_model import AutoModel

cfg = AutoConfig.for_model("deep_ar")  # matches the PF DeepAR reference
print(
    "hidden_size:",
    cfg.hidden_size,
    "| rnn_layers:",
    cfg.rnn_layers,
    "| embedding_size:",
    cfg.embedding_size,
    "| n_series:",
    cfg.n_series,
)

model = AutoModel.from_config(cfg, predict_sequence_length=PREDICTION_LENGTH)
inputs = [
    tf.keras.Input(shape=(ENCODER_LENGTH, 1), name="x"),
    tf.keras.Input(shape=(PREDICTION_LENGTH, 1), name="decoder_feature"),
    tf.keras.Input(shape=(1,), dtype="int32", name="static"),
]
km = model.build_model(inputs)
km.summary(line_length=90)

## 4. Train (negative log-likelihood)
DeepAR is probabilistic, so we train with a custom step that minimizes the Gaussian NLL of the true target under the predicted `(loc, scale)`, summed over the decoder horizon — the equivalent of `pytorch_forecasting`'s `NormalDistributionLoss`. Gradient clipping (`0.1`) mirrors the reference and keeps training stable.

In [ ]:
optimizer = tf.keras.optimizers.Adam(learning_rate=1e-2)


def train_step(xb, db, sb, yb):
    with tf.GradientTape() as tape:
        out = km([xb, db, sb], training=True)
        loc, scale = out["loc"], out["scale"]
        loss = tf.reduce_mean(0.5 * ((yb - loc) / scale) ** 2 + tf.math.log(scale) + 0.5 * LOG2PI)
    grads = tape.gradient(loss, km.trainable_variables)
    grads, _ = tf.clip_by_global_norm(grads, 0.1)
    optimizer.apply_gradients(zip(grads, km.trainable_variables))
    return loss


train_ds = (
    tf.data.Dataset.from_tensor_slices((train_batch.x, train_batch.decoder_feature, train_batch.static, train_batch.y))
    .shuffle(5000, seed=42)
    .batch(BATCH_SIZE)
)

for ep in range(1, 13):
    losses = []
    for xb, db, sb, yb in train_ds:
        losses.append(float(train_step(xb, db, sb, yb)))
    print(f"epoch {ep:3d}  train NLL = {np.mean(losses):.4f}")

## 5. Evaluate (ancestral sampling)
At inference DeepAR draws **ancestral samples**: each decoder step samples from the predicted Normal and feeds the *sampled* value back as the next lagged input. The point forecast is the **mean over `n_samples` paths** (matches `predict(mode='prediction', n_samples=100)`), un-normalized back to the raw value scale.

In [ ]:
def ancestral_sampling_point_forecast(n_samples=100, seed=0):
    B = val_batch.x.shape[0]
    samples = np.zeros((B, n_samples, PREDICTION_LENGTH), dtype=np.float32)
    for i0 in range(0, B, 10):
        i1 = min(i0 + 10, B)
        nb = i1 - i0
        xc = np.repeat(val_batch.x[i0:i1], n_samples, axis=0).astype(np.float32)
        sc = np.repeat(val_batch.static[i0:i1], n_samples, axis=0).astype(np.int64)
        dec = np.zeros((nb * n_samples, PREDICTION_LENGTH, 1), dtype=np.float32)
        dec[:, 0, 0] = xc[:, -1, 0]  # first decoder input = last encoder value
        for t in range(PREDICTION_LENGTH):
            out = km([xc, dec, sc], training=False)
            l = tf.squeeze(out["loc"][:, t, :]).numpy()
            s = tf.squeeze(out["scale"][:, t, :]).numpy()
            z = np.random.default_rng(seed + t).normal(loc=l, scale=s)
            if t + 1 < PREDICTION_LENGTH:
                dec[:, t + 1, 0] = z  # ancestral feedback of the sampled value
            samples[i0:i1, :, t] = z.reshape(nb, n_samples)
    return proc._inverse(samples).mean(axis=1)  # (B, PREDICTION_LENGTH) raw scale


preds = ancestral_sampling_point_forecast(n_samples=100)
actual = val_batch.target_original[..., 0]
err = actual - preds
print("MAE:", float(np.abs(err).mean()))
print("MSE:", float(np.mean(err**2)))

## 6. Plot a forecast

In [ ]:
s = 0  # pick a series
window = val_batch.x[s, :, 0]  # normalized history (encoder window)
fut = np.arange(ENCODER_LENGTH, ENCODER_LENGTH + PREDICTION_LENGTH)

plt.figure(figsize=(10, 4))
plt.plot(np.arange(-ENCODER_LENGTH, 0), window, label="history")
plt.plot(fut, actual[s], label="actual", marker="o")
plt.plot(fut, preds[s], label="forecast (mean of 100 sampled paths)", marker="x")
plt.axvline(0, color="gray", ls="--")
plt.xlabel("time (relative to forecast start)")
plt.ylabel("value")
plt.legend()
plt.title(f"Series {s}: {PREDICTION_LENGTH}-step DeepAR forecast")
plt.tight_layout()
plt.show()